In [1]:
import numpy as np
from utils import train, train_withoutX, load_dataset, f_w_v, dev_sigmoid_c, dev_f_w_v
from methods import ECI, ECI_full, PID_log, PID_log_half_smooth, PID_log_half_smooth_bis, PID_log_full_smooth
import matplotlib.pyplot as plt 
import os 
import pickle 
import statistics

c:\Users\tdupuy\Documents\Online_CP\Relevance Aware Thresholding in OCP for TS\utils.py:5: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
def exe(dataset, start_train, end_train, start_test, end_test, basemodel, name_method, alpha, is_X,sliding=True,**kwargs):
    train_size=end_train-start_train+1
    #methods = {"ACI":ACI, "OGD":OGD, "SF_OGD":SF_OGD, "decay_OGD":decay_OGD, "ECI":ECI, "PID_log":PID_log, "PID_log_half_smooth":PID_log_half_smooth, "PID_log_full_smooth":PID_log_full_smooth}
    value_X = 'with_X' if is_X else 'without_X'
    value_slide = 'slide' if sliding else 'no_slide'
    result_path = f'./results/forecasts/{dataset}/{value_X}/{basemodel}/'
    config_name = f'{start_train}_{end_train}_{start_test}_{end_test}_{value_slide}'
    os.makedirs(result_path, exist_ok=True)
    file_name = result_path+config_name+'.pkl'
    try:
        with open(file_name, 'rb') as handle:
            forecasts = pickle.load(handle)
    except:
        data=load_dataset(dataset)
        Y=data['Y']
        Y_array=Y.to_numpy()
        if is_X:
            name_X=data.columns[:-1]
            X=data[name_X]
            X_array=X.to_numpy()
            y_test, y_pred= train(X_array, Y_array, start_train, end_train, start_test, end_test, basemodel, **kwargs)
            forecasts={'y_test':y_test,'y_pred':y_pred}
        else: 
            y_test, y_pred = train_withoutX(Y_array, start_train, end_train, start_test, end_test, basemodel, sliding, **kwargs)
            forecasts={'y_test':y_test,'y_pred':y_pred}
        with open(file_name, 'wb') as handle:
            pickle.dump(forecasts,handle)
    y_test, y_pred = forecasts['y_test'], forecasts['y_pred']

    if name_method=="ECI":
        y_lowers, y_uppers, err=ECI(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['f_dev'])
    elif name_method=="ECI_full":
        y_lowers, y_uppers, err=ECI_full(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['f'],kwargs['f_dev'])
    elif name_method=="PID_log":
        y_lowers, y_uppers, err=PID_log(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['Csat'], kwargs['KI'])
    elif name_method=="PID_log_half_smooth":
        y_lowers, y_uppers, err, smooth_err=PID_log_half_smooth(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['Csat'], kwargs['KI'], kwargs['f'])
    elif name_method=="PID_log_half_smooth_bis":
        y_lowers, y_uppers, err, smooth_err=PID_log_half_smooth_bis(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['Csat'], kwargs['KI'], kwargs['f'])
    elif name_method=="PID_log_full_smooth": 
        y_lowers, y_uppers, err, smooth_err =PID_log_full_smooth(y_test, y_pred, alpha, kwargs['q1'], kwargs['lr'], kwargs['Csat'], kwargs['KI'], kwargs['f'])
    else:
        raise ValueError("Unknown method")
    try:
        result={"y_lowers":y_lowers, "y_uppers":y_uppers, "err":err, 'smooth_err':smooth_err}
    except:
        result={"y_lowers":y_lowers, "y_uppers":y_uppers, "err":err}
    return result

def plot_exp(filename,results,name_methods,method_to_title):
    color_plot = {'coverage': '#377eb8','size': '#4daf4a'}
    fig, axs = plt.subplots(2,len(name_methods), figsize=(6,6))
    widths=[(results[method]['y_uppers']-results[method]['y_lowers']) for method in results.keys()]
    min_width = np.ceil(np.stack(widths).min()) if np.stack(widths).min()>=0 else np.floor(np.stack(widths).min()) 
    max_width = np.ceil(np.stack(widths).max()) if np.stack(widths).max()>=0 else np.floor(np.stack(widths).max()) 
    for i,method in enumerate(results.keys()):
        cov=1-results[method]['err']
        cov_moving_avg=[cov[:i].sum()/len(cov[:i]) for i in range(1,len(cov)+1)]
        width=results[method]['y_uppers']-results[method]['y_lowers']
        axs[0,i].scatter(range(len(width)),width,c=color_plot['size'],marker='.',s=5)
        axs[1,i].scatter(range(len(cov_moving_avg)),cov_moving_avg,c=color_plot['coverage'],marker='.',s=5)
        axs[0,i].set_ylim(min_width*0.9, max_width*1.1)
        axs[1,i].set_ylim(-0.05,1.05)
        axs[0,i].set_title(f'{method_to_title[method]}')
    axs[0,0].set_ylabel('Size')
    axs[1,0].set_ylabel('Coverage')
    fig.supxlabel('Time')
    #fig.legend(loc="upper right")
    fig.tight_layout(rect=[0, 0, 1, 0.85])
    os.makedirs('./results/images', exist_ok=True)
    fig.savefig(f'results/images/{filename}.svg',bbox_inches='tight')
    print('Figure saved')
    plt.close()

def means_exp(result):
    cov=1-result['err']
    cov_mean=cov.mean()
    width=result['y_uppers']-result['y_lowers']
    width_mean=width.sum()/len(width)
    width_median=statistics.median(width)

    return cov_mean,width_mean, width_median

#### Experience 1: PID vs modified PID 


##### Temperature

In [11]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'AR'
is_X=False

alpha=0.1    
w=np.array([1])
v=np.array([17])

name_methods = ["PID_log","PID_log_half_smooth_bis"] 
results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting PID log


100%|██████████| 1210/1210 [00:00<00:00, 146197.73it/s]


End
0.9016528925619834 9.408454257266479 9.521653173793592
Starting PID log half smooth bis


100%|██████████| 1210/1210 [00:00<00:00, 25886.28it/s]

End
0.9041322314049587 9.044848136301935 8.829174098175168


Figure saved


In [12]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'Theta'
is_X=False

alpha=0.1    
w=np.array([1])
v=np.array([3])

name_methods = ["PID_log","PID_log_half_smooth_bis"] 
results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha, is_X, period_regressor=365,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)
    
method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


  0%|          | 0/1210 [00:00<?, ?it/s]

100%|██████████| 1210/1210 [00:20<00:00, 57.78it/s]


Forecasting finished
Starting PID log


100%|██████████| 1210/1210 [00:00<00:00, 147416.50it/s]


End
0.9024793388429752 10.87989893112195 9.395162430277608
Starting PID log half smooth bis


100%|██████████| 1210/1210 [00:00<00:00, 39297.45it/s]


End
0.9057851239669421 7.379919239360545 7.299401099771048
Figure saved


##### Amazon stock price dataset

In [13]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([16]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [00:04<00:00, 641.94it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 286058.56it/s]


End
0.9039186134137152 32.41947984794342 27.053226685266168
Starting PID log half smooth bis


  0%|          | 0/2654 [00:00<?, ?it/s]c:\Users\tdupuy\Documents\Online_CP\Relevance Aware Thresholding in OCP for TS\utils.py:130: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
100%|██████████| 2654/2654 [00:00<00:00, 32604.65it/s]


End
0.9039186134137152 29.361519274714436 22.40569989221416
Figure saved


In [14]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([17]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [01:00<00:00, 44.09it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 151071.22it/s]


End
0.892991710625471 23.167909920904606 19.540968906661206
Starting PID log half smooth bis


  0%|          | 0/2654 [00:00<?, ?it/s]c:\Users\tdupuy\Documents\Online_CP\Relevance Aware Thresholding in OCP for TS\utils.py:130: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
100%|██████████| 2654/2654 [00:00<00:00, 34818.61it/s]


End
0.9050489826676714 27.276996287773006 19.098599751761554
Figure saved


##### Microsoft stock price dataset

In [15]:
dataset = "MSFT" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([6]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [00:04<00:00, 560.04it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 110612.23it/s]


End
0.9012810851544838 4.503211751059348 3.5615230719526316
Starting PID log half smooth bis


100%|██████████| 2654/2654 [00:00<00:00, 22764.04it/s]

End
0.9009042954031651 3.6977119471495663 2.9085925432531994


Figure saved


In [16]:
dataset = "MSFT" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([2]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [00:56<00:00, 47.37it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 90754.57it/s]


End
0.8948756593820648 3.9960456136622886 2.9732156715718077
Starting PID log half smooth bis


100%|██████████| 2654/2654 [00:00<00:00, 34899.39it/s]


End
0.8990203466465713 3.9117400151714152 2.808088157941379
Figure saved


##### Google stock price dataset

In [17]:
dataset = "GOOGL" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([5]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [00:04<00:00, 608.20it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 109134.15it/s]


End
0.9076865109269028 43.75130458800765 39.439563461597686
Starting PID log half smooth bis


100%|██████████| 2654/2654 [00:00<00:00, 28292.93it/s]

End
0.9016578749058025 34.96140982274231 35.14432093265546


Figure saved


In [18]:
dataset = "GOOGL" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'  
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([1]) 
name_methods = ["PID_log","PID_log_half_smooth_bis"]
is_X=False

results=dict.fromkeys(name_methods)
for method in name_methods:
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.005,Csat=5,KI=10,f=f_w_v(w,v,alpha))
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)


method_to_title={"PID_log":"PID","PID_log_half_smooth_bis":"modified PID"}
title=f'exp1_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Forecasting...


100%|██████████| 2654/2654 [00:56<00:00, 47.13it/s]


Forecasting finished
Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 148707.96it/s]


End
0.9076865109269028 38.61440861085284 31.973157001341292
Starting PID log half smooth bis


  0%|          | 0/2654 [00:00<?, ?it/s]c:\Users\tdupuy\Documents\Online_CP\Relevance Aware Thresholding in OCP for TS\utils.py:130: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
100%|██████████| 2654/2654 [00:00<00:00, 29684.25it/s]


End
0.9076865109269028 35.47637686211633 27.853912719800093
Figure saved


#### Experience 2: ECI vs modified ECI

##### Temperature

In [19]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'AR'
is_X=False

alpha=0.1    
w=np.array([1])
v=np.array([5]) 
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.1,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 1210/1210 [00:00<00:00, 79670.77it/s]


End
0.7801652892561983 3.731165974418718 3.764343286366344
Starting ECI


100%|██████████| 1210/1210 [00:00<00:00, 33946.53it/s]

End
0.8975206611570248 5.323355975840122 5.37517724000349


Figure saved


In [20]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'Theta'
is_X=False

alpha=0.1    
w=np.array([1])
v=np.array([5]) 
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=365,q1=0,lr=0.1,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 1210/1210 [00:00<00:00, 100784.57it/s]


End
0.7793388429752066 3.7091814564478205 3.776298819902955
Starting ECI


100%|██████████| 1210/1210 [00:00<00:00, 34259.32it/s]

End
0.8950413223140495 5.408618728986191 5.49441245281518


Figure saved


##### Amazon stock price dataset

In [21]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([4])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.5,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 115746.44it/s]


End
0.825923134890731 14.941710842803722 11.078795043494608
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 32078.48it/s]

End
0.8997739261492087 17.674359282521856 14.322575418343462


Figure saved


In [22]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([4])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.5,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 100458.29it/s]


End
0.8244159758854559 14.772405327012224 11.024421487733008
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 33462.24it/s]

End
0.89939713639789 17.499838653372638 14.284751440982546


Figure saved


##### Microsoft stock price dataset

In [23]:
dataset = "MSFT" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'

alpha=0.1   
w=np.array([1])
v=np.array([3])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.05,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 78089.67it/s]


End
0.8289374529012811 1.3209576229916737 1.2868538595041947
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 30708.94it/s]

End
0.9005275056518462 1.7235632944881896 1.5756078907544193


Figure saved


In [24]:
dataset = "MSFT" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([3])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.05,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 74682.72it/s]


End
0.8296910324039186 1.3114307405793135 1.266130657215811
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 25144.75it/s]

End
0.9016578749058025 1.7015715788165944 1.6185933046032872


Figure saved


##### Google stock price dataset

In [25]:
dataset = "GOOGL" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([3])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,q1=0,lr=0.5,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 82321.54it/s]


End
0.8440090429540317 17.341668435258764 15.501054473270983
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 36358.92it/s]

End
0.8990203466465713 20.09024667518197 18.48586879130363


Figure saved


In [26]:
dataset = "GOOGL" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'
is_X=False

alpha=0.1   
w=np.array([1])
v=np.array([3])  
method="ECI"
name_methods = ["ECI_sig","ECI_my"]
functions=[dev_sigmoid_c(1),dev_f_w_v(w,v,alpha)]
results=dict.fromkeys(name_methods)
for f,method_f in zip(functions,name_methods):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha,is_X,period_regressor=5,q1=0,lr=0.5,f_dev=f)
    results[method_f]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)

method_to_title={"ECI_sig":"ECI","ECI_my":"modified ECI"}
title=f'exp2_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 107017.92it/s]


End
0.8440090429540317 17.516423156688436 15.320248765425504
Starting ECI


100%|██████████| 2654/2654 [00:00<00:00, 29583.19it/s]

End
0.89939713639789 20.053224196387454 18.219361123865852


Figure saved


#### Experience 3: PID vs PID half smooth vs PID full smooth 

##### Temperature

In [27]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'AR'
is_X=False

alpha=0.1    
w=np.array([1])
name_methods = ["PID_log", "PID_log_half_smooth", "PID_log_full_smooth"]
functions=['',f_w_v(w,np.array([69]),alpha),f_w_v(w,np.array([67]),alpha)]
results=dict.fromkeys(name_methods)
for method,function in zip(name_methods,functions):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha, is_X,q1=0,lr=0.005,Csat=5,KI=10,f=function)
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)
    
method_to_title={"PID_log":"PID","PID_log_half_smooth":"PID half smooth","PID_log_full_smooth":"PID full smooth" }
title=f'exp3_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting PID log


100%|██████████| 1210/1210 [00:00<00:00, 76548.78it/s]


End
0.9016528925619834 9.408454257266479 9.521653173793592
Starting PID log half smooth


100%|██████████| 1210/1210 [00:00<00:00, 27354.50it/s]


End
0.8966942148760331 7.872170881142151 7.671747289458246
Starting PID log full smooth


100%|██████████| 1210/1210 [00:00<00:00, 20780.89it/s]

End
0.8909090909090909 8.058196830877328 7.201238310553396


Figure saved


In [28]:
dataset = "daily-climate" #pas de ligne avec indice 1461
start_train = 1
end_train = 365
start_test = 366
end_test = 1575
basemodel= 'Theta'
is_X=False

alpha=0.1    
w=np.array([1])
name_methods = ["PID_log", "PID_log_half_smooth", "PID_log_full_smooth"]
functions=['',f_w_v(w,np.array([63]),alpha),f_w_v(w,np.array([75]),alpha)]
results=dict.fromkeys(name_methods)
for method,function in zip(name_methods,functions):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha, is_X,period_regressor=365,q1=0,lr=0.005,Csat=5,KI=10,f=function)
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)
    
method_to_title={"PID_log":"PID","PID_log_half_smooth":"PID half smooth","PID_log_full_smooth":"PID full smooth" }
title=f'exp3_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting PID log


100%|██████████| 1210/1210 [00:00<00:00, 118339.50it/s]


End
0.9024793388429752 10.87989893112195 9.395162430277608
Starting PID log half smooth


100%|██████████| 1210/1210 [00:00<00:00, 33010.12it/s]


End
0.8950413223140495 7.627862384878948 7.731940441145374
Starting PID log full smooth


100%|██████████| 1210/1210 [00:00<00:00, 37235.38it/s]

End
0.8958677685950414 9.701724738002659 8.21668439589018


Figure saved


##### Amazon stock price dataset 

In [29]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'AR'
is_X=False

alpha=0.1    
w=np.array([1])
name_methods = ["PID_log", "PID_log_half_smooth", "PID_log_full_smooth"]
functions=['',f_w_v(w,np.array([26]),alpha),f_w_v(w,np.array([26]),alpha)]
results=dict.fromkeys(name_methods)
for method,function in zip(name_methods,functions):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha, is_X,q1=0,lr=0.005,Csat=5,KI=10,f=function)
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)
    
method_to_title={"PID_log":"PID","PID_log_half_smooth":"PID half smooth","PID_log_full_smooth":"PID full smooth" }
title=f'exp3_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 68131.61it/s]


End
0.9039186134137152 32.41947984794342 27.053226685266168
Starting PID log half smooth


100%|██████████| 2654/2654 [00:00<00:00, 33337.57it/s]


End
0.8933685003767897 25.206166383233967 18.4157225222433
Starting PID log full smooth


100%|██████████| 2654/2654 [00:00<00:00, 26045.67it/s]

End
0.892991710625471 23.80324282047633 18.686945988651672


Figure saved


In [30]:
dataset = "AMZN" 
start_train = 1
end_train = 365
start_test = 366
end_test = 3019
basemodel= 'Theta'
is_X=False

alpha=0.1    
w=np.array([1])
name_methods = ["PID_log", "PID_log_half_smooth", "PID_log_full_smooth"]
functions=['',f_w_v(w,np.array([23]),alpha),f_w_v(w,np.array([25]),alpha)]
results=dict.fromkeys(name_methods)
for method,function in zip(name_methods,functions):
    result=exe(dataset, start_train, end_train, start_test, end_test, basemodel, method, alpha, is_X,period_regressor=5,q1=0,lr=0.005,Csat=5,KI=10,f=function)
    results[method]=result
    cov,mean_width,median_width=means_exp(result)
    print(cov,mean_width,median_width)
    
method_to_title={"PID_log":"PID","PID_log_half_smooth":"PID half smooth","PID_log_full_smooth":"PID full smooth" }
title=f'exp3_{dataset}_{basemodel}'
plot_exp(title,results,name_methods,method_to_title)

Starting PID log


100%|██████████| 2654/2654 [00:00<00:00, 103708.75it/s]


End
0.892991710625471 23.167909920904606 19.540968906661206
Starting PID log half smooth


100%|██████████| 2654/2654 [00:00<00:00, 33249.45it/s]


End
0.8892238131122834 31.955309747909556 19.53739780277975
Starting PID log full smooth


100%|██████████| 2654/2654 [00:00<00:00, 29150.22it/s]

End
0.8914845516201959 29.602905709581204 22.454189164837928


Figure saved
